# 3 · Naive Bayes & KNN

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(0)

## Bayes kuralı: satranç örneği

In [2]:
prior = np.array([0.5, 0.25, 0.25])      # P(Tip i)
win = np.array([0.3, 0.4, 0.5])          # P(Win | Tip i)
p_win = np.sum(prior * win)              # toplam olasılık
posterior = prior * win / p_win          # Bayes
print("P(Win) =", p_win, " P(Tip|Win) =", posterior.round(3))

P(Win) = 0.375  P(Tip|Win) = [0.4   0.267 0.333]


## Naive Bayes — elle ve sklearn ile (tenis verisi)

In [3]:
tennis = pd.DataFrame([
 ["Sunny","Hot","High","Weak","No"],["Sunny","Hot","High","Strong","No"],["Overcast","Hot","High","Weak","Yes"],
 ["Rain","Mild","High","Weak","Yes"],["Rain","Cool","Normal","Weak","Yes"],["Rain","Cool","Normal","Strong","No"],
 ["Overcast","Cool","Normal","Strong","Yes"],["Sunny","Mild","High","Weak","No"],["Sunny","Cool","Normal","Weak","Yes"],
 ["Rain","Mild","Normal","Weak","Yes"],["Sunny","Mild","Normal","Strong","Yes"],["Overcast","Mild","High","Strong","Yes"],
 ["Overcast","Hot","Normal","Weak","Yes"],["Rain","Mild","High","Strong","No"]],
 columns=["Outlook","Temp","Humidity","Wind","Play"])
feats = ["Outlook","Temp","Humidity","Wind"]
x_new = {"Outlook":"Sunny","Temp":"Cool","Humidity":"High","Wind":"Strong"}

prior = tennis["Play"].value_counts(normalize=True)
scores = {}
for k in prior.index:
    sub = tennis[tennis["Play"] == k]
    s = prior[k]
    for c, v in x_new.items():
        s *= (sub[c] == v).mean()          # P(x_j = v | Y = k)
    scores[k] = s
tot = sum(scores.values())
print("Skorlar:", scores)
print("Posterior:", {k: round(v / tot, 3) for k, v in scores.items()})

Skorlar: {'Yes': np.float64(0.005291005291005291), 'No': np.float64(0.02057142857142857)}
Posterior: {'Yes': np.float64(0.205), 'No': np.float64(0.795)}


In [4]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder()
Xc = enc.fit_transform(tennis[feats])
cnb = CategoricalNB(alpha=1e-10).fit(Xc, tennis["Play"])     # alpha~0: smoothing yok (elle hesapla aynı)
q = enc.transform(pd.DataFrame([x_new]))
print(cnb.classes_, cnb.predict_proba(q).round(3))
cnb_lap = CategoricalNB(alpha=1).fit(Xc, tennis["Play"])      # Laplace
print("Laplace ile:", cnb_lap.predict_proba(q).round(3))

['No' 'Yes'] [[0.795 0.205]]
Laplace ile: [[0.72 0.28]]


## Gaussian Naive Bayes ve KNN (sayısal veri)

In [5]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=600, n_features=2, n_redundant=0, n_informative=2,
                           n_clusters_per_class=1, class_sep=1.0, flip_y=0.08, random_state=1)
X[:, 1] *= 1000                                    # farklı ölçek: ölçeklemenin önemini görmek için
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

gnb = GaussianNB().fit(X_train, y_train)
print("GaussianNB acc:", accuracy_score(y_test, gnb.predict(X_test)))
print("Sınıf ortalamaları:\n", gnb.theta_)

knn_raw = KNeighborsClassifier(5).fit(X_train, y_train)
sc = StandardScaler().fit(X_train)
knn = KNeighborsClassifier(5).fit(sc.transform(X_train), y_train)
print("KNN ölçeksiz acc:", accuracy_score(y_test, knn_raw.predict(X_test)))
print("KNN ölçekli  acc:", accuracy_score(y_test, knn.predict(sc.transform(X_test))))

GaussianNB acc: 0.9166666666666666
Sınıf ortalamaları:
 [[ 9.68295967e-01  9.82141826e+02]
 [-9.44728367e-01  1.02072945e+03]]
KNN ölçeksiz acc: 0.5166666666666667
KNN ölçekli  acc: 0.9083333333333333


In [6]:
Ks = range(1, 51)
tr_err, te_err = [], []
Xtr, Xte = sc.transform(X_train), sc.transform(X_test)
for k in Ks:
    m = KNeighborsClassifier(k).fit(Xtr, y_train)
    tr_err.append(1 - m.score(Xtr, y_train)); te_err.append(1 - m.score(Xte, y_test))
plt.plot([1 / k for k in Ks], tr_err, ".-", label="Train error")
plt.plot([1 / k for k in Ks], te_err, ".-", label="Test error")
plt.xscale("log"); plt.xlabel("1/K (esneklik)"); plt.ylabel("Hata oranı"); plt.legend(); plt.show()
print("En iyi K:", list(Ks)[int(np.argmin(te_err))])

En iyi K: 4
